 # Pillar 1: SDE Drift Modeling, Lipschitz Bounds & Asymptotic Stability

 This notebook validates:
 1. **Hypothesis H1 (Variance Ceiling):** Spectral Normalization ($L_\theta \le 1.0$) stabilizes per-pixel error variance $\text{Var}(\delta_k) \le \frac{\sigma_0^2}{1 - L_\theta^2}$ as $k \to \infty$, preventing exponential divergence $\mathcal{O}(L_\theta^{2k})$.
 2. **Hypothesis H5 (Analytical Schedule Superiority):** Physics-derived noise $\sigma_{\text{noise}}(k)$ and heat diffusion blur $\sigma_{\text{blur}}(k)$ stabilize autoregressive rollouts over fixed heuristic corruptions.

In [ ]:
# Install repository package
# %pip install "brec @ git+https://github.com/CV-RL-Deep/MRI-Brain-Reconstruction.git@feature/pillar-1"

In [ ]:
import os
import gc
import copy
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from tqdm import tqdm

from brec.core.env import KAGGLE, configure_xla_paths
from brec.configs.config import CFG, Config

# --- Environment Path Initialization ---
if KAGGLE:
    print("🚀 Kaggle environment detected. Setting dataset & artifact paths...")
    os.environ.setdefault(
        'PATH_DATA_IXI', '/kaggle/input/preprocessed-oasis-and-epilepsy-and-ixi'
    )
    os.environ.setdefault(
        'PATH_DATA_BRATS',
        '/kaggle/input/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData',
    )
    os.environ.setdefault(
        'WEIGHTS_DIR',
        '/kaggle/input/brain-reconstruction-pretrain/ablations-B0',
    )
    os.environ.setdefault('RESULTS_DIR', 'pillar1_results')
    os.environ.setdefault('FIGURES_DIR', 'pillar1_figures')

    CFG.data.data_root_ixi = os.environ['PATH_DATA_IXI']
    CFG.data.data_root_brats = os.environ['PATH_DATA_BRATS']
    CFG.data.weights_dir = os.environ['WEIGHTS_DIR']
    CFG.data.results_dir = os.environ['RESULTS_DIR']
    CFG.data.figures_dir = os.environ['FIGURES_DIR']
else:
    print("💻 Local/Cluster environment detected.")
    configure_xla_paths()

os.makedirs(CFG.data.results_dir, exist_ok=True)
os.makedirs(CFG.data.figures_dir, exist_ok=True)

In [ ]:
from brec.core.utils import logger, InferenceProfiler
from brec.core.geometry import GeometryOps
from brec.data.files import find_t1_files, get_brats_subjects
from brec.data.cache import ActiveLoader, StaticLoader, VolumeLoader
from brec.data.generators import (
    IXIActiveGenerator,
    BraTSActiveGenerator,
    SequentialValidationGenerator,
    create_tf_dataset,
    get_training_dataset,
)
from brec.models.builder import ModelBuilder
from brec.models.losses import reset_perceptual_loss
from brec.training.trainer import Trainer
from brec.inference.reconstructor import VolumeReconstructor
from brec.evaluation.metrics import Pillar1Diagnostics, calculate_region_metrics
from brec.utils import calculate_step_metrics

 ### Phase 1: Empirical Parameter Calibration ($\sigma_0, D_{\text{spatial}}$)

In [ ]:
logger.info(
    ">>> STEP 1: Calibrating Physics & Innovation Constants from Clean Dataset <<<"
)

t1_files = find_t1_files(CFG.data.data_root_ixi)
brats_list = get_brats_subjects(CFG.data.data_root_brats)

# Load a subset of 15 volumes to compute spatial Laplacian diffusion D_spatial
sample_vols = []
for p in t1_files[:15]:
    v = VolumeLoader.load(p)
    if v is not None:
        sample_vols.append(v.t1)

estimated_d_spatial = Pillar1Diagnostics.estimate_spatial_diffusion_coefficient(
    volumes=sample_vols, delta_z=1.0
)

# Set estimated physics constant globally
CFG.aug.d_spatial = estimated_d_spatial
CFG.aug.delta_z = 1.0
CFG.aug.lipschitz_bound = 0.95
CFG.aug.sigma_0 = 0.015  # Will be refined upon training base model

logger.info(f"Calibrated D_spatial = {CFG.aug.d_spatial:.6f}")

 ### Phase 2: Pillar 1 Ablation Study Matrix

In [ ]:
def get_pillar1_ablation_configs(base_cfg: Config) -> dict[str, Config]:
    """
    Constructs the 4 experimental branches for Pillar 1:
    1. baseline_spade_full: SPADE + Buffer + PE (L_theta > 1.0 unconstrained, heuristic corruptions)
    2. p1_spectral_only   : SPADE + Spectral Norm (L_theta <= 1.0), heuristic corruptions
    3. p1_analytical_only : Unconstrained L_theta, analytical physics schedules (sigma_noise(k), sigma_blur(k))
    4. p1_full            : Spectral Norm (L_theta <= 1.0) + Analytical schedules (Full Pillar 1)
    """
    configs = {}

    # Anchor Model: SPADE + Buffer + PE (Heuristic noise/blur, no Spectral Normalization)
    cfg_base = copy.deepcopy(base_cfg)
    cfg_base.model.architecture = 'spade'
    cfg_base.model.use_positional_encoding = True
    cfg_base.aug.prob_hallucination_max = 0.65
    cfg_base.model.use_spectral_norm = False
    cfg_base.aug.use_analytical_schedules = False
    configs['baseline_spade_full'] = cfg_base

    # Branch A: Spectral Normalization Only
    cfg_sn = copy.deepcopy(cfg_base)
    cfg_sn.model.use_spectral_norm = True
    cfg_sn.model.spectral_norm_iterations = 1
    cfg_sn.aug.use_analytical_schedules = False
    configs['p1_spectral_only'] = cfg_sn

    # Branch B: Analytical Physics Schedules Only
    cfg_ana = copy.deepcopy(cfg_base)
    cfg_ana.model.use_spectral_norm = False
    cfg_ana.aug.use_analytical_schedules = True
    cfg_ana.aug.d_spatial = base_cfg.aug.d_spatial
    configs['p1_analytical_only'] = cfg_ana

    # Branch C: Full Pillar 1 (Spectral Norm + Analytical Schedules)
    cfg_full_p1 = copy.deepcopy(base_cfg)
    cfg_full_p1.model.architecture = 'spade'
    cfg_full_p1.model.use_positional_encoding = True
    cfg_full_p1.aug.prob_hallucination_max = 0.65
    cfg_full_p1.model.use_spectral_norm = True
    cfg_full_p1.model.spectral_norm_iterations = 1
    cfg_full_p1.aug.use_analytical_schedules = True
    cfg_full_p1.aug.d_spatial = base_cfg.aug.d_spatial
    configs['p1_full'] = cfg_full_p1

    return configs

 ### Phase 3: Train & Run Long-Horizon Rollouts ($k \ge 40$)

In [ ]:
def run_pillar1_ablation_study(base_config: Config):
    configs = get_pillar1_ablation_configs(base_config)
    results_dir = base_config.data.results_dir

    t1_files = find_t1_files(base_config.data.data_root_ixi)
    brats_list = get_brats_subjects(base_config.data.data_root_brats)

    random.seed(42)
    random.shuffle(t1_files)
    ixi_split = int(0.9 * len(t1_files))
    train_files, val_files = t1_files[:ixi_split], t1_files[ixi_split:]

    random.shuffle(brats_list)
    brats_split = int(0.9 * len(brats_list))
    train_brats, val_brats = brats_list[:brats_split], brats_list[brats_split:]

    is_interactive = base_config.run_type == 'Interactive'
    n_eval_vols = 2 if is_interactive else 8
    rollout_span = (
        10 if is_interactive else 40
    )  # 40 steps for asymptotic variance proof

    test_ixi = val_files[:n_eval_vols]
    val_loader = StaticLoader(base_config, val_files[:20], val_brats[:20])

    variance_metrics = []
    step_metrics = []
    lipschitz_records = []

    for ab_name, cfg in configs.items():
        logger.info(
            f"\n======================================================="
        )
        logger.info(f"   PILLAR 1 ABLATION: {ab_name.upper()}")
        logger.info(f"=======================================================")

        local_weights_path = os.path.join(results_dir, f"model_{ab_name}.keras")

        with ActiveLoader(cfg, train_files, train_brats) as train_loader:
            model = ModelBuilder.build(cfg)
            train_loader.model = model

            if os.path.exists(local_weights_path):
                logger.info(
                    f"✅ Found pre-trained weights for {ab_name}. Loading..."
                )
                model.load_weights(local_weights_path)
            else:
                logger.info(f"⚙️ Training {ab_name}...")
                train_ds = get_training_dataset(
                    IXIActiveGenerator(train_loader),
                    BraTSActiveGenerator(train_loader, mode='clean'),
                    cfg,
                )
                val_gen = SequentialValidationGenerator(
                    val_loader, max_slices_per_vol=15
                )
                val_ds = create_tf_dataset(val_gen, cfg, is_training=False)

                train_steps = (
                    max(10, (len(train_files) * 130) // cfg.train.batch_size)
                    if is_interactive
                    else max(
                        50, (len(train_files) * 130) // cfg.train.batch_size
                    )
                )

                trainer = Trainer(
                    cfg,
                    model,
                    train_ds,
                    val_ds,
                    train_loader,
                    train_steps=train_steps,
                    val_steps=None,
                )
                trainer.train(compile_model=True)
                model.save(local_weights_path)

            train_loader.stop()

            # --- Empirical Lipschitz Constant Measurement ---
            sample_val_batch = next(
                iter(create_tf_dataset(val_gen, cfg, is_training=False).take(1))
            )[0]
            l_hat = Pillar1Diagnostics.estimate_empirical_lipschitz(
                model, sample_val_batch
            )
            lipschitz_records.append(
                {'Ablation': ab_name, 'Empirical_Lipschitz': l_hat}
            )

            # --- Long-Horizon Rollout Inference (k = 40) ---
            reconstructor = VolumeReconstructor(model, cfg)
            reconstructor.cfg.batch_mode = True

            for vol_obj in tqdm(
                [v for v in val_loader.pool['ixi'] if v.path in test_ixi],
                desc=f"Rollouts ({ab_name})",
            ):
                vol = vol_obj.t1
                vol_id = (
                    os.path.basename(vol_obj.path)
                    .replace('.nii.gz', '')
                    .replace('.nii', '')
                )

                center = vol.shape[0] // 2
                start, end = center - rollout_span, center + rollout_span
                if start < 0 or end > vol.shape[0]:
                    continue

                recon_vol = reconstructor.autoregressive_restore(
                    vol, start, end, direction='forward', mask_volume=None
                )

                # Collect variance progression Var(delta_k)
                var_records = (
                    Pillar1Diagnostics.compute_stepwise_error_variance(
                        gt_vol=vol,
                        pred_vol=recon_vol,
                        start_idx=start,
                        end_idx=end,
                    )
                )
                for r in var_records:
                    r['Ablation'] = ab_name
                    r['Volume_ID'] = vol_id
                    variance_metrics.append(r)

                # Standard step metrics (SSIM, PSNR, MAE, FFL)
                s_metrics = calculate_step_metrics(
                    gt_vol=vol,
                    pred_vol=recon_vol,
                    start_idx=start,
                    end_idx=end,
                    direction='forward',
                    ablation_name=ab_name,
                    vol_id=vol_id,
                )
                step_metrics.extend(s_metrics)

        del model
        reset_perceptual_loss()
        gc.collect()
        tf.keras.backend.clear_session()
        gc.collect()

    # Save DataFrames
    df_var = pd.DataFrame(variance_metrics)
    df_var.to_csv(
        os.path.join(results_dir, "pillar1_variance_metrics.csv"), index=False
    )

    df_step = pd.DataFrame(step_metrics)
    df_step.to_csv(
        os.path.join(results_dir, "pillar1_step_metrics.csv"), index=False
    )

    df_lip = pd.DataFrame(lipschitz_records)
    df_lip.to_csv(
        os.path.join(results_dir, "pillar1_lipschitz_metrics.csv"), index=False
    )

    logger.info("✅ Pillar 1 Experiments Completed. Data saved to disk.")
    return df_var, df_step, df_lip

In [ ]:
df_var, df_step, df_lip = run_pillar1_ablation_study(CFG)

 ### Phase 4: Verification of Asymptotic Error Variance ($H_1$)

In [ ]:
logger.info(">>> Plotting Fig P1: Asymptotic Error Variance Stabilization <<<")

PALETTE_P1 = {
    'baseline_spade_full': '#E63946',  # Red (Exponential growth)
    'p1_spectral_only': '#F4A261',  # Orange
    'p1_analytical_only': '#2A9D8F',  # Teal
    'p1_full': '#1D3557',  # Dark Navy Blue (Asymptotic ceiling)
}

LABELS_P1 = {
    'baseline_spade_full': 'Unconstrained Baseline (L > 1.0)',
    'p1_spectral_only': '+ Spectral Norm (L ≤ 1.0)',
    'p1_analytical_only': '+ Analytical SDE Schedule',
    'p1_full': 'Full Pillar 1 (Spectral + SDE Physics)',
}

fig, axes = plt.subplots(1, 2, figsize=(10, 4.0))

# 1. Error Variance Curve Var(delta_k) vs k
sns.lineplot(
    data=df_var,
    x='Rollout_Step_k',
    y='Error_Variance',
    hue='Ablation',
    palette=PALETTE_P1,
    linewidth=2.0,
    ax=axes[0],
)
axes[0].set_title(
    r"$\mathbf{H_1:}$ Error Variance Ceiling $\operatorname{Var}(\delta_k)$",
    fontsize=10,
)
axes[0].set_xlabel(r"Autoregressive Rollout Step ($k$)")
axes[0].set_ylabel(r"Spatial Error Variance $\operatorname{Var}(\delta_k)$")
axes[0].grid(True, linestyle='--', alpha=0.5)

# 2. Structural Similarity Decay SSIM vs k
sns.lineplot(
    data=df_step,
    x='Rollout_Step_k',
    y='SSIM',
    hue='Ablation',
    palette=PALETTE_P1,
    linewidth=2.0,
    ax=axes[1],
)
axes[1].set_title(
    r"$\mathbf{H_5:}$ Structural Contiguity ($\operatorname{SSIM}$)",
    fontsize=10,
)
axes[1].set_xlabel(r"Autoregressive Rollout Step ($k$)")
axes[1].set_ylabel(r"Structural Similarity ($\operatorname{SSIM}$)")
axes[1].grid(True, linestyle='--', alpha=0.5)

handles, labels = axes[0].get_legend_handles_labels()
axes[0].get_legend().remove()
axes[1].get_legend().remove()

fig.legend(
    handles,
    [LABELS_P1.get(l, l) for l in labels],
    loc='lower center',
    bbox_to_anchor=(0.5, -0.08),
    ncol=2,
    fontsize=9,
    framealpha=0.85,
)

plt.tight_layout()
fig_path = os.path.join(
    CFG.data.figures_dir, "Fig_Pillar1_Variance_Asymptotics.pdf"
)
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()

logger.info(f"📊 Saved Pillar 1 Validation Figure to {fig_path}")

In [ ]:
# Display Empirical Lipschitz Ratios
print("=== Empirical Lipschitz Constant (Operator Norm Ratio) ===")
print(df_lip.to_string(index=False))